In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import ollama  # Replaces vLLM
from ollama import AsyncClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# =====================================================
# INITIALIZATION & BATCH LOADING
# =====================================================

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

In [3]:
# 3. Initialize database
print("Loading database for documents...")
vectorstore = Chroma(
    collection_name="rag_vietnamese_docs",
    persist_directory="../process_data/my_chroma_db",
    embedding_function=embeddings
)
print("Documents:", vectorstore._collection.count())

Loading database for documents...
Documents: 662071


In [4]:
# Load the Reranker Model (FlagReranker supports BGE/Qwen-based rerankers efficiently)
from FlagEmbedding import FlagReranker
print("Loading Reranker Model...")
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

Loading Reranker Model...


In [5]:
import json
from ollama import AsyncClient

# =====================================================
# PHASE 1: QWEN HYBRID ROUTER
# =====================================================

async def qwen_classify(question: str) -> str:
    prompt = (
            "Bạn là một chuyên gia phân loại câu hỏi hệ thống. "
            "Hãy phân loại câu hỏi dưới đây vào 1 trong 2 nhóm:\n\n"
            "1. 'MATH': CHỈ dành cho các bài toán thuần túy (số học, đại số, giải tích, hình học, giải phương trình toán học cơ bản) KHÔNG đòi hỏi kiến thức chuyên ngành hay công thức vật lý/hóa học.\n"
            "2. 'KNOWLEDGE': Dành cho mọi câu hỏi khác, BAO GỒM cả Vật lý, Hóa học, Kinh tế học (để tra cứu định lý, hằng số, công thức), và các yêu cầu tra cứu thông tin, sự kiện, luật, lịch sử.\n\n"
            "Chỉ trả về đúng MỘT từ là 'MATH' hoặc 'KNOWLEDGE'. Tuyệt đối không giải thích thêm."
        )
    
    client = AsyncClient()
    response = await client.chat(
        model='qwen2.5:7b',
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': f"Câu hỏi: {question}"}
        ],
        options={'temperature': 0.0} # 0.0 for absolute deterministic routing
    )
    
    result = response['message']['content'].strip().upper()
    if "MATH" in result:
        return "MATH", 0.95
    return "KNOWLEDGE", 0.90

async def hybrid_router(question: str) -> tuple[str, float]:
    """
    Evaluates the question using O(1) string matching for Reading Comprehension,
    then falls back to Qwen for Math vs Knowledge routing.
    """
    question_lower = question.lower()

    # 1. Reading Comprehension O(1) Check (Unchanged)
    if any(kw in question_lower for kw in ["đoạn thông tin:", "tiêu đề:", "title:", "nội dung:"]):
        return "READING", 1.0000

    # 2. LLM-based Classification
    return await qwen_classify(question)

In [6]:
async def generate_search_queries(question: str, choices: str) -> list[str]:
    """Generates a list of search queries using Qwen, strictly enforcing a JSON Object format."""
    
    prompt = (
        "Bạn là một chuyên gia tìm kiếm ngữ nghĩa (Semantic Search). "
        "Nhiệm vụ của bạn là tạo ra từ 1 đến 3 câu truy vấn ngắn gọn, tập trung vào các khái niệm cốt lõi "
        "để trích xuất tài liệu chính xác từ cơ sở dữ liệu nhằm trả lời câu hỏi dưới đây.\n\n"
        f"Câu hỏi: {question}\n"
        f"Các lựa chọn:\n{choices}\n\n"
        "Yêu cầu:\n"
        "- Trả về MỘT OBJECT JSON chứa một mảng các chuỗi truy vấn với key là 'queries'.\n"
        "- Ví dụ: {\"queries\": [\"từ khóa 1\", \"từ khóa 2\"]}\n"
        "- Không giải thích thêm."
    )
    
    client = AsyncClient()
    
    response = await client.chat(
        model='qwen2.5:7b',
        messages=[{'role': 'user', 'content': prompt}],
        format='json', 
        options={
            'temperature': 0.1, 
            'num_predict': 150
        }
    )
    
    raw_text = response['message']['content'].strip()
    
    try:
        # Parse the JSON object
        data = json.loads(raw_text)
        
        # Extract the list from the "queries" key
        if "queries" in data and isinstance(data["queries"], list):
            # Clean up the strings and ensure it's not empty
            queries = [str(q).strip() for q in data["queries"] if str(q).strip()]
            if queries:
                print(f"Good queries: '{queries}'")
                return queries
                
    except json.JSONDecodeError:
        pass
        
    print(f"⚠️ Query parsing failed. LLM Output: '{raw_text}'")
    return [question]

In [7]:
import asyncio
import uuid
from FlagEmbedding import FlagReranker
import math

# =====================================================
# PHASE 2 INITIALIZATION
# =====================================================
# Async Queue for low-confidence queries that need rewriting
async_retry_queue = asyncio.Queue()

# =====================================================
# CORE RAG FUNCTIONS
# =====================================================

async def retrieve_top_k(queries: list[str], k_per_query: int = 5) -> list[str]:
    """Retrieves documents for a list of queries and deduplicates them."""
    all_docs = []
    seen_content = set()
    
    for q in queries:
        docs = vectorstore.similarity_search(q, k=k_per_query)
        for doc in docs:
            # Deduplicate based on content
            if doc.page_content not in seen_content:
                seen_content.add(doc.page_content)
                all_docs.append(doc)
                
    return all_docs


async def rerank_documents(query: str, docs: list[str], top_n: int = 3) -> list[tuple[str, float]]:
    """
    Uses the Cross-Encoder Reranker to score the retrieved documents against the query
    and returns the top N documents with their confidence scores.
    """
    if not docs:
        return []
        
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.compute_score(pairs)
    
    # Pair documents with their scores and sort descending
    doc_scores = list(zip(docs, scores))
    doc_scores.sort(key=lambda x: x[1], reverse=True)
    
    return doc_scores[:top_n]

async def rewrite_query(query: str, choices: str = "") -> str:
    prompt = (
        "Bạn là một chuyên gia hỗ trợ tìm kiếm ngữ nghĩa. Hãy viết lại câu hỏi dưới đây thành các từ khóa "
        "tối ưu nhất để tra cứu trong cơ sở dữ liệu tài liệu.\n"
        "MẸO QUAN TRỌNG: Nếu nhận thấy đây là câu hỏi Vật lý, Hóa học hoặc Kinh tế học, "
        "hãy tự động bổ sung các từ khóa như 'công thức', 'định luật', 'hằng số', hoặc 'phản ứng' "
        "để hệ thống có thể tìm đúng công thức cần thiết.\n\n"
        "Ví dụ minh họa:\n"
        "Câu hỏi: Một vật rơi tự do từ độ cao h. Biểu thức tính vận tốc của vật khi chạm đất là gì?\n"
        "Output: [\"công thức tính vận tốc rơi tự do\", \"định luật bảo toàn cơ năng vật rơi chạm đất\"]\n"
        f"Câu hỏi: {query}\n"  # BUG FIX: Changed {question} to {query} to match the function parameter
        f"Các lựa chọn:\n{choices}\n\n"
        "Output:"
    )

    client = AsyncClient()
    
    # UPDATE: Swapped client.generate for client.chat
    response = await client.chat(
        model='qwen2.5:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'temperature': 0.1, # Decreased from 0.3 to reduce the chance of the LLM going off-script
            'num_predict': 100
        }
    )
    
    # UPDATE: Changed response extraction to match the chat endpoint structure
    return response['message']['content'].strip()

# =====================================================
# PHASE 2: PRECISION RAG PIPELINE
# =====================================================
def sigmoid(x):
    return 1 / (1 + math.exp(-x))



async def phase2_precision_rag(original_query: str, search_queries: list[str], format_answer: str = "", choices: str = "", is_retry: bool = False):
    # 1. Retrieve using the MULTIPLE generated queries
    retrieved_docs = await retrieve_top_k(search_queries, k_per_query=8)

    # 2. Rerank the aggregated documents against the ORIGINAL query
    reranked_results = await rerank_documents(original_query, retrieved_docs, top_n=5)

    # 3. Retrieval Confidence Check
    top_doc, top_raw_score = reranked_results[0] if reranked_results else ("", -999)    
    top_score_prob = sigmoid(top_raw_score) if top_raw_score != -999 else 0.0
    CONFIDENCE_THRESHOLD = 0.70

    if top_score_prob < CONFIDENCE_THRESHOLD and not is_retry:
        print(f"⚠️ Low confidence ({top_score_prob:.2f}). Queuing for rewrite: '{original_query}'")
        rewritten_query = await rewrite_query(original_query, choices)
        
        loop = asyncio.get_running_loop()
        reply_future = loop.create_future()
        
        await async_retry_queue.put({
            "original_query": original_query, 
            "rewritten_query": rewritten_query, 
            "format_answer": format_answer,
            "future": reply_future  
        })
        
        return "", "QUEUED_FOR_RETRY", reply_future

    final_context = "\n".join([f"- {doc.page_content}" for doc, score in reranked_results])
    return final_context, "SUCCESS", None

In [8]:
async def phase3_reasoning(question: str, context_or_flag: str, format_answer: str) -> str:
    # 1. Determine the System Instruction
    if context_or_flag == "MATH":
        system_msg = (
            "Bạn là một chuyên gia học thuật xuất sắc. Nhiệm vụ của bạn là giải quyết bài toán này một cách chính xác tuyệt đối. "
            "BẠN PHẢI TUÂN THỦ NGHIÊM NGẶT QUY TRÌNH SAU:\n"
            "1. ĐỊNH NGHĨA: Trích xuất các dữ kiện và viết ra công thức chuẩn xác trước khi tính.\n"
            "2. TÍNH TOÁN TỪNG BƯỚC: Thực hiện MỖI phép tính đại số trên một dòng riêng biệt. Bắt buộc viết rõ các bước biến đổi phân số, nhân chia hoặc đạo hàm. Tuyệt đối không làm gộp.\n"
            "3. KIỂM TRA LẠI: So sánh cẩn thận kết quả cuối cùng với các lựa chọn được cung cấp."
        )        
        user_msg = f"Câu hỏi: {question}\n\n{format_answer}"
        
    elif context_or_flag == "READING":
        system_msg = "Bạn là một trợ lý chuyên gia về đọc hiểu. Hãy phân tích đoạn văn bản và trả lời chính xác."
        user_msg = f"Câu hỏi: {question}\n\n{format_answer}"
        
    else:
        # UPDATE: Hybrid RAG System Message (Factual Q&A + Contextual Math)
        system_msg = (
            "Bạn là một trợ lý AI chuyên gia. Ưu tiên TUYỆT ĐỐI của bạn là sử dụng ngữ cảnh được cung cấp để trả lời.\n"
            "QUAN TRỌNG: Nếu câu hỏi yêu cầu tính toán (ví dụ: Vật lý, Hóa học, Kinh tế), hãy trích xuất công thức/hằng số "
            "từ ngữ cảnh và thực hiện giải toán từng bước (step-by-step) vô cùng logic.\n"
            "ĐỐI VỚI CÂU HỎI KIẾN THỨC VÀ PHÁP LÝ: Tuyệt đối không được tự bịa ra quy trình hợp lý. Nếu ngữ cảnh chứa thông tin "
            "khớp với một trong các lựa chọn, hãy chọn đáp án đó. Nếu không tìm thấy thông tin trực tiếp, hãy tìm các từ khóa đồng nghĩa trong ngữ cảnh."
        )
        user_msg = f"Ngữ cảnh:\n{context_or_flag}\n\nCâu hỏi: {question}\n\n{format_answer}"

    format_constraint = (
        "Bạn PHẢI cấu trúc câu trả lời của mình CHÍNH XÁC bằng các thẻ XML sau. "
        "CHỈ sử dụng tiếng Việt. Tuyệt đối KHÔNG sử dụng ngôn ngữ nào khác. "
        "Không được thêm bất kỳ văn bản nào bên ngoài các thẻ này:\n"
        "<answer_reasoning>\n[Suy luận logic từng bước của bạn]\n</answer_reasoning>\n"
        "<verification_check>\n[Tự đánh giá lại quá trình suy luận]\n</verification_check>\n"
        "<final_answer>\n[CHỈ ghi MỘT chữ cái duy nhất. Ví dụ: A, B, C, D...]\n</final_answer>"
    )

    client = AsyncClient()

    current_temp = 0.0 if context_or_flag == "MATH" else 0.1
    
    # Use client.chat instead of client.generate
    response = await client.chat(
        model='qwen2.5:7b',
        messages=[
            {'role': 'system', 'content': f"{system_msg}\n{format_constraint}"},
            {'role': 'user', 'content': user_msg}
        ],
        options={
            'temperature': current_temp, # Lowered from 0.2 to reduce hallucination in MCQs
            'top_p': 0.9,
            'num_predict': 2048,
            'repeat_penalty': 1.05,
            'stop': ['</final_answer>']
        }
    )
    return response['message']['content'].strip()

In [9]:
import re
import asyncio
import pandas as pd

# =====================================================
# PHASE 4: DETERMINISTIC OUTPUT LAYER
# =====================================================

def extract_and_format_answer(raw_llm_output: str) -> str:
    match = re.search(
            r'<final_answer>\s*(.*?)(?:</final_answer>|$)',
            raw_llm_output,
            re.DOTALL | re.IGNORECASE
        )

    if not match:
        return raw_llm_output

    extracted_text = match.group(1).strip().upper()

    letter_match = re.search(r'\b([A-Z])\b', extracted_text)

    return letter_match.group(1) if letter_match else extracted_text



# def extract_and_format_answer(raw_llm_output: str) -> str:
#     """
#     Extracts the content from <final_answer> tags.
#     Returns the full string exactly as the model wrote it.
#     """
#     match = re.search(r'<final_answer>\s*(.*?)\s*</final_answer>', raw_llm_output, re.DOTALL | re.IGNORECASE)
    
#     if not match:
#         return "UNKNOWN" # Fallback if model fails to follow format
        
#     # Return the exact text, just stripping leading/trailing whitespace
#     return match.group(1).strip()

In [10]:
# import asyncio

# # =====================================================
# # PHASE 4: ORCHESTRATION & QUEUE WORKER
# # =====================================================

# async def process_user_query(user_query: str) -> str:
#     """
#     The main entry point for a user's question.
#     Routes the query, retrieves context if necessary, and generates an answer.
#     """
#     print(f"\n--- Processing Query: '{user_query}' ---")
    
#     # -----------------------------------------
#     # PHASE 1: HYBRID ROUTING
#     # -----------------------------------------
#     category, confidence = await hybrid_router(user_query)
#     print(f"🚦 Route: [{category}] (Confidence: {confidence:.4f})")

#     # -----------------------------------------
#     # PHASE 2 & 3: RAG + REASONING
#     # -----------------------------------------
#     if category in ["MATH", "READING"]:
#         # Bypass Retrieval: Send directly to Qwen with the category flag
#         print("⚡ Bypassing RAG. Sending directly to Reasoning Engine...")
#         response = await phase3_reasoning(user_query, context_or_flag=category)
#         return response

#     elif category == "KNOWLEDGE":
#         # Execute Precision RAG
#         print("🔍 Executing Precision RAG...")
#         context, status = await phase2_precision_rag(user_query)

#         if status == "SUCCESS":
#             # Confidence was high enough, generate answer based on retrieved context
#             print("✅ RAG Successful. Generating response...")
#             response = await phase3_reasoning(user_query, context_or_flag=context)
#             return response
            
#         elif status == "QUEUED_FOR_RETRY":
#             # The query was confusing and got sent to the rewrite queue
#             return (
#                 "Tôi đang phân tích chuyên sâu câu hỏi của bạn vì nó khá phức tạp. "
#                 "Hệ thống đang xử lý ngầm và sẽ trả kết quả sớm nhất."
#             )

# async def background_queue_worker():
#     """
#     A background daemon that continuously processes low-confidence 
#     queries that were rewritten and queued.
#     """
#     print("🛠️ Background Queue Worker started...")
#     while True:
#         # Wait for an item to appear in the queue
#         task = await async_retry_queue.get()
#         orig_query = task["original_query"]
#         rewritten_query = task["rewritten_query"]
        
#         print(f"\n🔄 [BACKGROUND WORKER] Retrying Query: '{orig_query}'")
#         print(f"📝 Rewritten as: '{rewritten_query}'")

#         # Rerun Phase 2 with the rewritten query (set is_retry=True to avoid infinite loops)
#         context, status = await phase2_precision_rag(rewritten_query, is_retry=True)

#         # Proceed to Phase 3 regardless of the new score, but use the best context we found
#         final_response = await phase3_reasoning(orig_query, context_or_flag=context)
        
#         print(f"✅ [BACKGROUND WORKER] Finished processing for: '{orig_query}'")
#         print(f"Result:\n{final_response}\n")

#         # Mark the task as done in the queue
#         async_retry_queue.task_done()

# # =====================================================
# # SYSTEM ENTRY POINT
# # =====================================================

# async def main():
#     # 1. Start the background worker as a concurrent task
#     worker_task = asyncio.create_task(background_queue_worker())

#     # 2. Simulate incoming user queries
#     # test_queries = [
#     #     "Tính đạo hàm của hàm số f(x) = x^2 + 3x",  # Should trigger MATH directly
#     #     "Đoạn thông tin: Anh ấy sinh năm 1990. Câu hỏi: Anh ấy sinh năm bao nhiêu?", # READ
#     #     "Bao nhiêu tuổi thì được lái xe máy ở việt nam?", # RAG (KNOWLEDGE)
#     # ]

#     test_queries = [
#         "bao nhiêu tuổi thì được lái xe ở việt nam"
#     ]

#     # Process queries concurrently
#     tasks = [process_user_query(q) for q in test_queries]
#     results = await asyncio.gather(*tasks)

#     # Print immediate results
#     for q, res in zip(test_queries, results):
#         print(f"\n[FINAL OUTPUT for '{q}']\n{res}")

#     # Wait for the background queue to finish processing any low-confidence retries
#     await async_retry_queue.join()
    
#     # Cancel the infinite background worker loop gracefully
#     worker_task.cancel()

# if __name__ == "__main__":
#     await main()

In [11]:
import asyncio

# =====================================================
# PHASE 4: ORCHESTRATION & QUEUE WORKER
# =====================================================

async def process_user_query(item: dict) -> str:
    qid = item.get("qid")
    question_text = item.get("question")
    choices = item.get("choices", [])

    formatted_choices = "\n".join([f"- {c}" for c in choices])
    full_query = (
        f"Các lựa chọn:\n{formatted_choices}\n\n"
        f"Yêu cầu: Hãy chọn đáp án đúng nhất. Trong thẻ <final_answer>, "
        f"hãy CHỈ viết một chữ cái duy nhất đại diện cho đáp án đúng (ví dụ: A, B, C, D, E, F, v.v.). "
        f"Không giải thích hay viết thêm bất kỳ văn bản nào khác."  
    )
    
    # 1. Call the new Qwen Router
    category, confidence = await hybrid_router(question_text)

    if category in ["MATH", "READING"]:
        return await phase3_reasoning(question_text, context_or_flag=category, format_answer=full_query)

    elif category == "KNOWLEDGE":
        # 2. Generate Search Queries using Qwen
        search_queries = await generate_search_queries(question=question_text, choices=formatted_choices)
        
        # 3. Pass the generated queries to Phase 2, but keep original_query for Reranking
        context, status, reply_future = await phase2_precision_rag(
            original_query=question_text, 
            search_queries=search_queries,
            choices = formatted_choices,
            format_answer=full_query
        )

        if status == "SUCCESS":
            return await phase3_reasoning(question_text, context_or_flag=context, format_answer=full_query)
            
        elif status == "QUEUED_FOR_RETRY":
            final_response = await reply_future 
            return final_response

async def background_queue_worker():
    while True:
        task = await async_retry_queue.get()
        reply_future = task["future"]
        
        try:
            orig_query = task["original_query"]
            rewritten_query = task["rewritten_query"]
            full_query = task["format_answer"]

            # FIX: Added search_queries parameter and passed full_query
            context, status, _ = await phase2_precision_rag(
                original_query=rewritten_query, 
                search_queries=[rewritten_query], 
                format_answer=full_query,
                is_retry=True
            )
            
            final_response = await phase3_reasoning(orig_query, context_or_flag=context, format_answer=full_query)

            if not reply_future.done():
                reply_future.set_result(final_response)
                
        except Exception as e:
            # If the worker crashes, send the error back so the main loop doesn't hang forever
            if not reply_future.done():
                reply_future.set_exception(e)
        finally:
            async_retry_queue.task_done()

In [12]:
import pandas as pd
import asyncio
from tqdm import tqdm

async def run_evaluation(dataset: list, output_csv: str = "evaluation_results.csv") -> pd.DataFrame:
    """
    Evaluates a dataset of questions, expecting the model to output the full string of the answer.
    """
    results = []
    
    # 1. Start the background worker for retries
    worker_task = asyncio.create_task(background_queue_worker())
    count = 0
    for item in tqdm(dataset):
        # 3. Run through your existing pipeline
        qid = item.get("qid")
        correct_answer = item.get("answer", "").strip()
        question = item.get("question", "")
        raw_response = await process_user_query(item)
        
        # 4. Extract the full string answer using the updated Phase 4 function
        model_answer = extract_and_format_answer(raw_response)

        model_answer = model_answer.replace("$", "").strip()
        correct_answer = correct_answer.replace("$", "").strip()

        model_answer = model_answer.replace("<", "").strip()
        model_answer = model_answer.replace(">", "").strip()
        
        correct_answer = correct_answer.rstrip(".")
        model_answer = model_answer.rstrip(".")
        
        # 5. Evaluate correctness (ignoring case and leading/trailing spaces for safety)
        is_true = (model_answer.lower() == correct_answer.lower())
        if is_true:
            count += 1
        
        # Store results
        results.append({
            "qid": qid,
            "question": question,
            "raw_response": raw_response,
            "model_answer": model_answer,
            "correct_answer": correct_answer,
            "is_true": is_true
        })

    # 6. Clean up
    worker_task.cancel()
    
    # 7. Export to CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig") 
    
    return df, count

In [15]:
import json
import time

with open('public-test-formatted.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)


print(f"Loaded {len(test_data)} questions from public-test.json")

start_time = time.perf_counter()

results_df, count = await run_evaluation(
    test_data,
    output_csv="model_evaluation_results8.csv"
)

end_time = time.perf_counter()

print(f"Execution time: {end_time - start_time:.2f} seconds")
print(f"Accuracy: {count/len(test_data):.2f}")


Loaded 463 questions from public-test.json


  1%|▏         | 6/463 [00:32<43:11,  5.67s/it]

Good queries: '['biểu hiện biến đổi khí hậu', 'mực nước biển dâng']'


  2%|▏         | 7/463 [00:35<38:18,  5.04s/it]

Good queries: '['tỷ lệ lạm phát GDP', 'chênh lệch giữa GDP danh nghĩa và thực tế']'


...

Good queries: '['mặt người mặt của']'
⚠️ Low confidence (0.02). Queuing for rewrite: 'Tìm câu tục ngữ có ý nghĩa giống với câu tục ngữ: “Một mặt người bằng mười mặt của”.'


100%|█████████▉| 462/463 [52:03<00:06,  6.80s/it]

Good queries: '['phương pháp giáo dục thuyết phục', 'mối quan hệ mật thiết', 'yếu tố ảnh hưởng']'
⚠️ Low confidence (0.07). Queuing for rewrite: 'Phương pháp giáo dục thuyết phục có mối quan hệ mật thiết với yếu tố nào?'


100%|██████████| 463/463 [52:13<00:00,  6.77s/it]

Execution time: 3133.87 seconds
Accuracy: 0.77


In [39]:
# correct_answer = "$ B(t) = B_0 e^{-kt} $"
# correct_answer = correct_answer.replace("$", "").strip()
# print(correct_answer)

In [101]:
# import json
# import string

# def format_multiple_choice(data):
#     letters = list(string.ascii_uppercase)

#     for item in data:
#         original_choices = item.get("choices", [])
#         original_answer = item.get("answer", "")

#         new_choices = []
#         new_answer = ""

#         for index, choice in enumerate(original_choices):
#             letter = letters[index]

#             if choice == original_answer:
#                 new_answer = letter

#             new_choices.append(f"{letter}. {choice}")

#         item["choices"] = new_choices
#         item["answer"] = new_answer

#     return data


# # Read
# with open("public-test.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# # Process
# formatted_data = format_multiple_choice(data)

# # Save
# with open("public-test-formatted.json", "w", encoding="utf-8") as f:
#     json.dump(formatted_data, f, ensure_ascii=False, indent=2)

# print("Saved to public-test-formatted.json")

Saved to public-test-formatted.json
